In [2]:
import warnings
from IPython.display import Markdown, display
warnings.filterwarnings("ignore")

In [3]:
import pdfplumber as pdf

In [4]:
filename = "Muhammad-pages.pdf"
dictionary_for_pages=[]
with pdf.open(f"../data/building/{filename}") as pdfFile:
    for pageNo,content in enumerate(pdfFile.pages):
        dictionary_for_pages.append({"pageNo":pageNo,"text" : content.extract_text()})
        


In [6]:
i=0
overlap=100
chunkSize=300

chunks=[]
words=""
for singleDictionary in dictionary_for_pages:
    words=singleDictionary["text"].split(" ")
    # because chunkNumber has to become 1 for every new page
    chunkNumber=1
    i=0
    while i < len(words):
        chunks.append(
            {"pageNo":singleDictionary["pageNo"],
             "chunkNumber":chunkNumber,
             "chunk_text" : " ".join(words[i:i+chunkSize])
            })
        i+=(chunkSize-overlap)
        chunkNumber+=1
        


In [33]:
chunks

[{'pageNo': 0,
  'chunkNumber': 1,
  'chunk_text': 'Abu Talib politely dismissed them at first, thinking it was just a heated talk. But as Muhammad grew\nmore vocal, Abu Talib requested Muhammad to not burden him beyond what he could bear, to which\nMuhammad wept and replied that he would not stop even if they put the sun in his right hand and the\nmoon in his left. When he turned around, Abu Talib called him and said, "Come back nephew, say what\nyou please, for by God I will never give you up on any account."[126][127]\nQuraysh delegation to Yathrib\nThe leaders of the Quraysh sent Nadr ibn al-Harith and Uqba ibn Abi Mu\'ayt to Yathrib to seek the\nopinions of the Jewish rabbis regarding Muhammad. The rabbis advised them to ask Muhammad three\nquestions: recount the tale of young men who ventured forth in the first age; narrate the story of a traveler\nwho reached both the eastern and western ends of the earth; and provide details about the spirit. If\nMuhammad answered correctly, th

In [8]:
from sentence_transformers import SentenceTransformer as ST
model = ST("all-MiniLM-L6-v2")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 360.17it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
chunksText=[]
for chunk in chunks:
    chunksText.append(chunk["chunk_text"])
chunksText
embeddings = model.encode(chunksText)

In [11]:
embeddings

array([[ 0.00812211,  0.14528619,  0.00533355, ..., -0.02064898,
        -0.02102001, -0.04554903],
       [-0.01362283,  0.13042772, -0.04900342, ..., -0.04167355,
        -0.01575064, -0.02756287],
       [-0.03895748,  0.05755593, -0.02950185, ..., -0.07731067,
        -0.01279373,  0.00839455],
       [ 0.00337972,  0.08561469, -0.05775772, ..., -0.04442552,
         0.00710447, -0.08361203],
       [-0.04821   ,  0.10012358, -0.02494188, ..., -0.05496677,
        -0.03818604, -0.06456722],
       [-0.04110324,  0.1094072 , -0.01041555, ..., -0.00425496,
        -0.03317889, -0.0469301 ]], shape=(6, 384), dtype=float32)

In [12]:
import faiss 

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [13]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../.env")
os.getenv("GROQ_API_KEY")
# how to create client groq thing
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
print(client)

In [28]:
def ask(query):
    message = (query + "\n Answer the above question,only using the text below ,and \
    say I don't know if not found ,and also if you do find an answer then also tell ,which page number and chunk number u used\n")
    for i in indices[0]:
        message +=f"Page Number = {chunks[i]["pageNo"]}\n"
        message +=f"Chunk Number = {chunks[i]["chunkNumber"]}\n"
        message += chunks[i]["chunk_text"] + "\n"
    messages = [{"role": "user", "content": message}]
    response  = client.chat.completions.create(model="llama-3.3-70b-versatile",messages=messages)
    return response

In [29]:
query1 = "What happened when Muhammad went to Ta'if?"
query_embeddings =  model.encode([query1])
distances,indices = index.search(query_embeddings,3)
response = ask(query1)
# irrelevant question,to test what it responds
query2 = "What's the capital of France?"
query_embeddings =  model.encode([query2])
distances,indices = index.search(query_embeddings,3)
response1 = ask(query2)

In [30]:
responseOfQuery = response.choices[0].message.content
responseOfQuery1 = response1.choices[0].message.content


In [31]:
display(Markdown(responseOfQuery))

When Muhammad went to Ta'if, he was met with rejection and hostility. The people of Ta'if said, "If you are truly a prophet, what need do you have of our help? If God sent you as his messenger, why doesn't He protect you? And if Allah wished to send a prophet, couldn't He have found a better person than you, a weak and fatherless orphan?" They pelted him with stones, injuring his limbs. Muhammad then escaped to the garden of Utbah ibn Rabi'ah.

I found this answer in Page Number = 1, Chunk Number = 2.

In [32]:
display(Markdown(responseOfQuery1))

I don't know. The text does not mention the capital of France. I have searched through all the provided chunks (Page Number = 1, Chunk Number = 1 and 2, and Page Number = 0, Chunk Number = 2) but could not find any information related to the capital of France.